In [1]:
import sys
sys.path.append("../src")
import pandas as pd
from area_eval import add_keys, predictions, evaluate

pd.set_option("display.max_colwidth", 60)
df = add_keys(pd.read_parquet("../data/processed/karachi_events_clean.parquet"))
gold = add_keys(pd.read_csv("../data/processed/area_test.csv"))
gold = gold.merge(df[["key", "eventid"]].drop_duplicates("key"), on="key", how="left")
print("Matched to events:", gold["eventid"].notna().sum(), "of", len(gold))

preds, known = predictions(df, "../data/processed/area_gliner_raw.parquet")
results = {n: evaluate(gold, p, n, known) for n, p in preds.items()}
print(pd.DataFrame([s for _, s in results.values()]).to_string(index=False))

for name, (r, _) in results.items():
    bad = r[(r.has_gold & ~r.correct) | (~r.has_gold & r.predicted)]
    print(f"\n{name}: mistakes")
    print(bad[["gold", "pred", "location"]].to_string())

Matched to events: 30 of 30
method  coverage  precision  recall  recall_recurring  accuracy
 rules      0.70       0.95    0.77              0.95      0.80
gliner      0.63       0.95    0.69              0.86      0.73
hybrid      0.70       0.95    0.77              0.95      0.80

rules: mistakes
                                  gold    pred                                                       location
4                               T Town  Saddar         Incident occurred in the Saddar Town area of the city.
13                               Lyari     NaN  The incident occurred in the Layari neighborhood of the city.
26                       Karachi court     NaN                                                            NaN
27                       Rehman Clinic     NaN                                                            NaN
28  Quaid-e-Azam international airport     NaN                                                            NaN
29                        Nipa flyover 